# Cross-Exchange Perpetual Futures Arbitrage: Kraken vs Coinbase

Screens 10 mid-cap perpetual futures pairs (same symbols as spot notebook) for basis spread divergence / convergence patterns that support delta-neutral cross-exchange arbitrage.

**Strategy logic (leverage account):**
- Basis = Kraken perp price − Coinbase perp price (typically tracked for convergence)
- When Kraken perp price > Coinbase perp: **long Coinbase, short Kraken**
- When Coinbase perp price > Kraken perp: **long Kraken, short Coinbase**
- Exit when basis reverts (converges)

**Fee model (worst-case taker/taker):**
- Kraken Futures taker: 0.05%
- Coinbase Perpetuals taker: ~0.10% (estimated, varies by tier)
- Total round-trip: ≈ 30–40 bps — **2.8×–5.7× better than spot-to-spot (172 bps)**

**Key difference from spot:** Funding rates paid/received on both sides typically net out, leaving only basis spread divergence as the profit source.

**Data:** 1-minute OHLC candles, last 2 weeks, cached as Parquet.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import display
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

In [2]:
# ── Date range: last 2 weeks ───────────────────────────────────────────────────
DATE_FROM_ISO = "2026-05-28T00:00:00+00:00"
DATE_TO_ISO   = "2026-06-11T00:00:00+00:00"

# ── Perpetual futures symbols (same bases as spot notebook) ─────────────────────
# Kraken Futures use PF_ prefix (USD-margined perpetuals)
# Coinbase Perpetuals use -USD-PERP suffix
SYMBOLS = [
    {"base": "LTC",  "kraken": "PF_LTCUSD",  "coinbase": "LTC-USD-PERP"},
    {"base": "LINK", "kraken": "PF_LINKUSD", "coinbase": "LINK-USD-PERP"},
    {"base": "BCH",  "kraken": "PF_BCHUSD",  "coinbase": "BCH-USD-PERP"},
    {"base": "ATOM", "kraken": "PF_ATOMUSD", "coinbase": "ATOM-USD-PERP"},
    {"base": "UNI",  "kraken": "PF_UNIUSD",  "coinbase": "UNI-USD-PERP"},
    {"base": "AAVE", "kraken": "PF_AAVEUSD", "coinbase": "AAVE-USD-PERP"},
    {"base": "DOT",  "kraken": "PF_DOTUSD",  "coinbase": "DOT-USD-PERP"},
    {"base": "ALGO", "kraken": "PF_ALGOUSD", "coinbase": "ALGO-USD-PERP"},
    {"base": "XLM",  "kraken": "PF_XLMUSD",  "coinbase": "XLM-USD-PERP"},
    {"base": "NEAR", "kraken": "PF_NEARUSD", "coinbase": "NEAR-USD-PERP"},
]

# ── Cache directory ────────────────────────────────────────────────────────────
_nb_dir = Path("notebooks/cross_exchange_kraken_coinbase")
DATA_DIR = _nb_dir / "data" if _nb_dir.exists() else Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Fee model ──────────────────────────────────────────────────────────────────
# Futures taker fees are much lower than spot
KRAKEN_FUTURES_TAKER_FEE   = 0.0005   # 0.05% perp taker
COINBASE_PERP_TAKER_FEE    = 0.0010   # 0.10% perp taker (conservative estimate)
ROUND_TRIP_BPS = (KRAKEN_FUTURES_TAKER_FEE + COINBASE_PERP_TAKER_FEE) * 2 * 10_000

print(f"Download window : {DATE_FROM_ISO} → {DATE_TO_ISO}")
print(f"Symbols         : {[s['base'] for s in SYMBOLS]}")
print(f"Round-trip fees : {ROUND_TRIP_BPS:.1f} bps  "
      f"(Kraken {KRAKEN_FUTURES_TAKER_FEE*10_000:.0f} + Coinbase {COINBASE_PERP_TAKER_FEE*10_000:.0f} bps, open+close)")
print(f"Cache directory : {DATA_DIR}")
print(f"\nNote: Spot comparison was {(0.0026+0.0060)*2*10_000:.0f} bps round-trip. Futures is {ROUND_TRIP_BPS/((0.0026+0.0060)*2*10_000):.1f}× cheaper.")

Download window : 2026-05-28T00:00:00+00:00 → 2026-06-11T00:00:00+00:00
Symbols         : ['LTC', 'LINK', 'BCH', 'ATOM', 'UNI', 'AAVE', 'DOT', 'ALGO', 'XLM', 'NEAR']
Round-trip fees : 30.0 bps  (Kraken 5 + Coinbase 10 bps, open+close)
Cache directory : notebooks/cross_exchange_kraken_coinbase/data

Note: Spot comparison was 172 bps round-trip. Futures is 0.2× cheaper.


In [3]:
# ── Timestamp helpers ──────────────────────────────────────────────────────────

def iso_to_ts(value: str) -> pd.Timestamp:
    return pd.Timestamp(value)

def iso_to_unix_s(value: str) -> int:
    return int(pd.Timestamp(value).timestamp())

def cache_path(symbol: str, exchange: str) -> Path:
    safe_start = DATE_FROM_ISO[:10]
    safe_end   = DATE_TO_ISO[:10]
    return DATA_DIR / f"{symbol}__{exchange}__perp__1m__{safe_start}__{safe_end}.parquet"


# ── HTTP sessions ──────────────────────────────────────────────────────────────

kraken_session = requests.Session()
kraken_session.headers.update({"User-Agent": "alphakit-cross-arb/0.1"})

cb_session = requests.Session()
cb_session.headers.update({"User-Agent": "alphakit-cross-arb/0.1"})

MAX_RETRIES = 5
RETRY_SLEEP = 2.0


def kraken_get(path: str, params: dict | None = None):
    url = f"https://futures.kraken.com/api/charts/v1/trade{path}"
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = kraken_session.get(url, params=params, timeout=30)
            if resp.status_code == 429:
                time.sleep(RETRY_SLEEP * (attempt + 1))
                continue
            resp.raise_for_status()
            return resp.json()
        except requests.HTTPError as exc:
            last_exc = exc
            if exc.response is not None and exc.response.status_code == 429:
                time.sleep(RETRY_SLEEP * (attempt + 1))
                continue
            raise
        except requests.RequestException as exc:
            last_exc = exc
            time.sleep(RETRY_SLEEP * (attempt + 1))
    if last_exc:
        raise last_exc
    raise RuntimeError("Kraken request failed")


def cb_get(path: str, params: dict | None = None):
    url = f"https://api.exchange.coinbase.com{path}"
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = cb_session.get(url, params=params, timeout=30)
            if resp.status_code == 429:
                time.sleep(RETRY_SLEEP * (attempt + 1))
                continue
            resp.raise_for_status()
            return resp.json()
        except requests.HTTPError as exc:
            last_exc = exc
            if exc.response is not None and exc.response.status_code == 429:
                time.sleep(RETRY_SLEEP * (attempt + 1))
                continue
            raise
        except requests.RequestException as exc:
            last_exc = exc
            time.sleep(RETRY_SLEEP * (attempt + 1))
    if last_exc:
        raise last_exc
    raise RuntimeError("Coinbase request failed")


print("Helpers loaded.")

Helpers loaded.


In [4]:
# ── Kraken Futures OHLC (public REST, no auth needed) ──────────────────────────
#
# Endpoint: GET https://futures.kraken.com/api/charts/v1/trade/{symbol}/{resolution}
# Response: {"candles": [{"time": ms, "open": float, "high": float, ...}], "more_candles": bool}

KRAKEN_FUTURES_DELAY = 0.2


def fetch_kraken_futures_1m(
    symbol: str,
    start_iso: str = DATE_FROM_ISO,
    end_iso:   str = DATE_TO_ISO,
) -> pd.DataFrame:
    from_s     = iso_to_unix_s(start_iso)
    to_s       = iso_to_unix_s(end_iso)
    interval_s = 60  # 1 minute in seconds

    all_rows: list[dict] = []
    cursor = from_s

    while cursor < to_s:
        try:
            resp = kraken_session.get(
                f"https://futures.kraken.com/api/charts/v1/trade/{symbol}/1m",
                params={"from": cursor, "to": to_s},
                timeout=30,
            )
            resp.raise_for_status()
        except requests.RequestException as exc:
            raise ValueError(
                f"Kraken Futures request failed for {symbol}: {exc}"
            ) from exc

        payload = resp.json()
        candles = payload.get("candles", [])
        if not candles:
            break

        for c in candles:
            ts_ms = int(c["time"])
            ts_s  = ts_ms // 1000
            if ts_s >= to_s:
                break
            all_rows.append({
                "timestamp": pd.Timestamp(ts_ms, unit="ms", tz="UTC"),
                "open":   float(c["open"]),
                "high":   float(c["high"]),
                "low":    float(c["low"]),
                "close":  float(c["close"]),
                "volume": float(c.get("volume", 0)),
            })

        if not payload.get("more_candles", False):
            break
        cursor = int(candles[-1]["time"]) // 1000 + interval_s
        time.sleep(KRAKEN_FUTURES_DELAY)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df = df[df["timestamp"] < iso_to_ts(end_iso)]
    df["exchange"] = "kraken"
    return (
        df.drop_duplicates("timestamp")
        .sort_values("timestamp")
        .reset_index(drop=True)
    )


print("Kraken Futures download function defined.")

Kraken Futures download function defined.


In [5]:
# ── Coinbase Perpetuals OHLC (via Exchange API) ────────────────────────────────
#
# Endpoint: GET /products/{product_id}/candles
# Note: Coinbase perp symbols end with -USD-PERP (e.g., LINK-USD-PERP, BTC-USD-PERP)
# If this endpoint doesn't work for perps, we may need to try the v3 API

CB_PERP_DELAY = 0.15
CB_MAX_CANDLES = 300


def fetch_coinbase_perp_1m(
    product_id: str,
    start_iso: str = DATE_FROM_ISO,
    end_iso:   str = DATE_TO_ISO,
) -> pd.DataFrame:
    start_ts = iso_to_unix_s(start_iso)
    end_ts   = iso_to_unix_s(end_iso)
    window_s = CB_MAX_CANDLES * 60   # 300 one-minute candles
    all_rows: list[dict] = []
    cursor   = start_ts

    while cursor < end_ts:
        batch_end = min(cursor + window_s, end_ts)
        try:
            data = cb_get(
                f"/products/{product_id}/candles",
                {
                    "granularity": 60,
                    "start": pd.Timestamp(cursor,    unit="s", tz="UTC").isoformat(),
                    "end":   pd.Timestamp(batch_end, unit="s", tz="UTC").isoformat(),
                },
            )
        except Exception as exc:
            print(f"      Warning: Coinbase perp {product_id} failed: {exc}")
            return pd.DataFrame()

        if isinstance(data, dict) and "message" in data:
            print(f"      Coinbase API error for {product_id}: {data['message']}")
            return pd.DataFrame()

        for c in data:
            ts_s = int(c[0])
            if start_ts <= ts_s < end_ts:
                all_rows.append({
                    "timestamp": pd.Timestamp(ts_s, unit="s", tz="UTC"),
                    "open":   float(c[3]),
                    "high":   float(c[2]),
                    "low":    float(c[1]),
                    "close":  float(c[4]),
                    "volume": float(c[5]),
                })

        cursor = batch_end
        time.sleep(CB_PERP_DELAY)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df["exchange"] = "coinbase"
    return (
        df.drop_duplicates("timestamp")
        .sort_values("timestamp")
        .reset_index(drop=True)
    )


print("Coinbase Perpetuals download function defined.")

Coinbase Perpetuals download function defined.


In [6]:
# ── Download (or load from cache) all perpetual symbols ────────────────────────

def load_or_download(base: str, exchange: str, symbol: str) -> pd.DataFrame:
    path = cache_path(symbol, exchange)
    if path.exists():
        df = pd.read_parquet(path)
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
        print(f"  [{exchange:8s}] {symbol:12s}: loaded from cache ({len(df):,} rows)")
        return df.sort_values("timestamp").reset_index(drop=True)

    print(f"  [{exchange:8s}] {symbol:12s}: downloading ...", end=" ", flush=True)
    if exchange == "kraken":
        df = fetch_kraken_futures_1m(symbol)
    else:
        df = fetch_coinbase_perp_1m(symbol)

    if df.empty:
        print("⚠️  no data")
        return pd.DataFrame()

    df.to_parquet(path, index=False)
    print(f"✅ {len(df):,} rows saved")
    return df


all_data: dict[str, dict[str, pd.DataFrame]] = {}

for sym in SYMBOLS:
    base = sym["base"]
    print(f"\n{base}:")
    kraken_df   = load_or_download(base, "kraken",   sym["kraken"])
    coinbase_df = load_or_download(base, "coinbase", sym["coinbase"])
    all_data[base] = {"kraken": kraken_df, "coinbase": coinbase_df}

print("\n✅ Download complete")


LTC:
  [kraken  ] PF_LTCUSD   : downloading ... ✅ 20,160 rows saved
  [coinbase] LTC-USD-PERP: downloading ...       Warning: Coinbase perp LTC-USD-PERP failed: 404 Client Error: Not Found for url: https://api.exchange.coinbase.com/products/LTC-USD-PERP/candles?granularity=60&start=2026-05-28T00%3A00%3A00%2B00%3A00&end=2026-05-28T05%3A00%3A00%2B00%3A00
⚠️  no data

LINK:
  [kraken  ] PF_LINKUSD  : downloading ... ✅ 20,160 rows saved
  [coinbase] LINK-USD-PERP: downloading ...       Warning: Coinbase perp LINK-USD-PERP failed: 404 Client Error: Not Found for url: https://api.exchange.coinbase.com/products/LINK-USD-PERP/candles?granularity=60&start=2026-05-28T00%3A00%3A00%2B00%3A00&end=2026-05-28T05%3A00%3A00%2B00%3A00
⚠️  no data

BCH:
  [kraken  ] PF_BCHUSD   : downloading ... ✅ 20,160 rows saved
  [coinbase] BCH-USD-PERP: downloading ...       Warning: Coinbase perp BCH-USD-PERP failed: 404 Client Error: Not Found for url: https://api.exchange.coinbase.com/products/BCH-USD-PERP/candl

In [7]:
# ── Merge data and compute basis spread ────────────────────────────────────────
#
# Basis = Kraken perp price - Coinbase perp price
# Positive basis: Kraken expensive → long CB / short Kraken
# Negative basis: Coinbase expensive → long Kraken / short CB

MIN_OVERLAP_ROWS = 5_000

pair_frames: dict[str, pd.DataFrame] = {}

for sym in SYMBOLS:
    base    = sym["base"]
    k_df    = all_data[base]["kraken"]
    c_df    = all_data[base]["coinbase"]

    if k_df.empty or c_df.empty:
        print(f"⚠️  {base}: skipping — missing data on one or both exchanges")
        continue

    merged = (
        k_df[["timestamp", "close", "volume"]]
        .rename(columns={"close": "kraken_close", "volume": "kraken_vol"})
        .merge(
            c_df[["timestamp", "close", "volume"]]
            .rename(columns={"close": "coinbase_close", "volume": "coinbase_vol"}),
            on="timestamp",
            how="inner",
        )
    )

    if len(merged) < MIN_OVERLAP_ROWS:
        print(f"⚠️  {base}: only {len(merged):,} overlapping rows — skipping")
        continue

    # Basis in bps: (Kraken - Coinbase) / Coinbase × 10,000
    merged["basis_bps"] = (
        (merged["kraken_close"] - merged["coinbase_close"])
        / merged["coinbase_close"]
        * 10_000
    )
    merged["base"] = base

    pair_frames[base] = merged.sort_values("timestamp").reset_index(drop=True)

print(f"\n✅ Built {len(pair_frames)} pair frames: {list(pair_frames.keys())}")

# Per-symbol basis statistics
basis_stats = []
for base, frame in pair_frames.items():
    b = frame["basis_bps"]
    basis_stats.append({
        "symbol":        base,
        "rows":          len(frame),
        "mean_bps":      round(b.mean(), 2),
        "std_bps":       round(b.std(), 2),
        "p1_bps":        round(b.quantile(0.01), 2),
        "p25_bps":       round(b.quantile(0.25), 2),
        "p75_bps":       round(b.quantile(0.75), 2),
        "p99_bps":       round(b.quantile(0.99), 2),
        "min_bps":       round(b.min(), 2),
        "max_bps":       round(b.max(), 2),
        "pct_above_rt_fee": round((b.abs() > ROUND_TRIP_BPS).mean() * 100, 2),
    })

basis_summary = pd.DataFrame(basis_stats).sort_values("std_bps", ascending=False)
print(f"\nBasis statistics (round-trip fee threshold: {ROUND_TRIP_BPS:.1f} bps):")
display(basis_summary)

⚠️  LTC: skipping — missing data on one or both exchanges
⚠️  LINK: skipping — missing data on one or both exchanges
⚠️  BCH: skipping — missing data on one or both exchanges
⚠️  ATOM: skipping — missing data on one or both exchanges
⚠️  UNI: skipping — missing data on one or both exchanges
⚠️  AAVE: skipping — missing data on one or both exchanges
⚠️  DOT: skipping — missing data on one or both exchanges
⚠️  ALGO: skipping — missing data on one or both exchanges
⚠️  XLM: skipping — missing data on one or both exchanges
⚠️  NEAR: skipping — missing data on one or both exchanges

✅ Built 0 pair frames: []


KeyError: 'std_bps'

In [ ]:
# ── Plot: basis time series for all symbols ────────────────────────────────────

n_pairs = len(pair_frames)
fig = make_subplots(
    rows=n_pairs, cols=1,
    shared_xaxes=True,
    subplot_titles=list(pair_frames.keys()),
    vertical_spacing=0.015,
)

for i, (base, frame) in enumerate(pair_frames.items(), start=1):
    fig.add_trace(
        go.Scatter(
            x=frame["timestamp"],
            y=frame["basis_bps"],
            mode="lines",
            name=base,
            line=dict(width=0.7),
            showlegend=False,
        ),
        row=i, col=1,
    )
    fig.add_hline(y=0, line_width=0.5, line_color="gray", line_dash="dot", row=i, col=1)
    fig.add_hline(
        y=ROUND_TRIP_BPS, line_width=1, line_color="crimson", line_dash="dash", row=i, col=1
    )
    fig.add_hline(
        y=-ROUND_TRIP_BPS, line_width=1, line_color="crimson", line_dash="dash", row=i, col=1
    )

fig.update_layout(
    height=220 * n_pairs,
    width=1200,
    template="plotly_white",
    title=(
        f"Kraken − Coinbase perpetual futures basis (bps)  |  "
        f"Red dashes = ±{ROUND_TRIP_BPS:.0f} bps round-trip fee threshold"
    ),
)
fig.show()

In [ ]:
# ── Price comparison: Kraken vs Coinbase perp prices (top 4 by basis volatility) ─

top4 = basis_summary.head(4)["symbol"].tolist()

fig2 = make_subplots(
    rows=len(top4), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{s}: Kraken vs Coinbase perp prices" for s in top4],
    vertical_spacing=0.04,
)

for i, base in enumerate(top4, start=1):
    frame = pair_frames[base]
    # Normalise to first price for visual comparison
    k_norm = frame["kraken_close"]   / frame["kraken_close"].iloc[0]
    c_norm = frame["coinbase_close"] / frame["coinbase_close"].iloc[0]

    fig2.add_trace(go.Scatter(
        x=frame["timestamp"], y=k_norm,
        mode="lines", name="Kraken", line=dict(width=0.8, color="royalblue"),
        showlegend=(i == 1),
    ), row=i, col=1)

    fig2.add_trace(go.Scatter(
        x=frame["timestamp"], y=c_norm,
        mode="lines", name="Coinbase", line=dict(width=0.8, color="darkorange"),
        showlegend=(i == 1),
    ), row=i, col=1)

fig2.update_layout(
    height=300 * len(top4),
    width=1200,
    template="plotly_white",
    title="Normalised perp price comparison (Kraken vs Coinbase) — top 4 by basis volatility",
)
fig2.show()

In [ ]:
# ── Mean-reversion analysis: does basis revert predictably? ────────────────────

mr_rows = []

for base, frame in pair_frames.items():
    b = frame["basis_bps"].dropna()

    lag1_ac = float(b.autocorr(lag=1))

    # OLS half-life of basis reversion
    delta_b = b.diff().dropna()
    b_lag   = b.shift(1).dropna()
    idx     = delta_b.index.intersection(b_lag.index)
    if len(idx) > 20:
        X = b_lag.loc[idx].values
        y = delta_b.loc[idx].values
        X_mat = np.column_stack([np.ones_like(X), X])
        beta, *_ = np.linalg.lstsq(X_mat, y, rcond=None)
        slope     = float(beta[1])
        half_life = -np.log(2) / slope if slope < 0 else np.nan
    else:
        half_life = np.nan

    # Profitability window: % of time basis magnitude > round-trip fees
    frac_profitable = float((b.abs() > ROUND_TRIP_BPS).mean())

    mr_rows.append({
        "symbol":           base,
        "lag1_autocorr":    round(lag1_ac, 4),
        "half_life_min":    round(half_life, 1) if not np.isnan(half_life) else np.nan,
        "mean_basis":       round(float(b.mean()), 2),
        "std_basis":        round(float(b.std()), 2),
        "pct_profitable":   round(frac_profitable * 100, 2),
        "range_bps":        round(float(b.max() - b.min()), 2),
    })

mr_df = (
    pd.DataFrame(mr_rows)
    .sort_values("half_life_min", na_position="last")
    .reset_index(drop=True)
)

print("Basis mean-reversion stats:")
print("  lag1_autocorr < 0  → basis is mean-reverting")
print("  half_life_min      → minutes for basis to revert halfway")
print(f"  pct_profitable     → % of time |basis| > {ROUND_TRIP_BPS:.0f} bps (round-trip fee)")
display(mr_df)

In [ ]:
# ── Rolling 24h basis z-score ──────────────────────────────────────────────────

ZSCORE_WINDOW = 24 * 60

top2 = basis_summary.head(2)["symbol"].tolist()

fig3 = make_subplots(
    rows=len(top2), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{s}: 24h rolling z-score of basis" for s in top2],
    vertical_spacing=0.06,
)

for i, base in enumerate(top2, start=1):
    frame  = pair_frames[base].copy()
    mu     = frame["basis_bps"].rolling(ZSCORE_WINDOW, min_periods=60).mean()
    sigma  = frame["basis_bps"].rolling(ZSCORE_WINDOW, min_periods=60).std()
    zscore = (frame["basis_bps"] - mu) / sigma.clip(lower=1e-8)

    fig3.add_trace(go.Scatter(
        x=frame["timestamp"], y=zscore,
        mode="lines", name=base, line=dict(width=0.7),
        showlegend=False,
    ), row=i, col=1)
    for level in (2, -2):
        fig3.add_hline(y=level, line_width=1, line_color="crimson", line_dash="dash", row=i, col=1)
    fig3.add_hline(y=0, line_width=0.5, line_color="gray", line_dash="dot", row=i, col=1)

fig3.update_layout(
    height=350 * len(top2), width=1200, template="plotly_white",
    title="24h rolling z-score of Kraken-Coinbase basis  (red dashes = ±2σ entry signal)",
)
fig3.show()

In [ ]:
# ── Basis arbitrage simulation ─────────────────────────────────────────────────
#
# Entry: |basis| > ENTRY_THRESHOLD  →  establish position
# Exit:  |basis| < EXIT_THRESHOLD   →  close position (basis converged)
#
# P&L = (entry_basis - exit_basis) × capital / 2 / 10_000 - round_trip_fees

ENTRY_THRESHOLD_BPS = ROUND_TRIP_BPS * 1.5
EXIT_THRESHOLD_BPS  = ROUND_TRIP_BPS * 0.25
STARTING_CAPITAL    = 10_000.0

print(f"Entry threshold : {ENTRY_THRESHOLD_BPS:.1f} bps ({ENTRY_THRESHOLD_BPS/ROUND_TRIP_BPS:.1f}× fees)")
print(f"Exit threshold  : {EXIT_THRESHOLD_BPS:.1f} bps")
print(f"Starting capital: ${STARTING_CAPITAL:,.0f}")
print(f"Round-trip fees : {ROUND_TRIP_BPS:.1f} bps (vs {172:.0f} bps for spot)")

sim_results: list[dict] = []
all_trades:  list[dict] = []

for base, frame in pair_frames.items():
    balance   = STARTING_CAPITAL
    position  = None   # None | "long_kraken" | "long_coinbase"
    entry_row = None
    trades:   list[dict] = []

    for idx, row in frame.iterrows():
        basis   = float(row["basis_bps"])
        is_last = idx == frame.index[-1]

        if position is None:
            if basis > ENTRY_THRESHOLD_BPS:
                position  = "long_coinbase"  # Kraken expensive → long CB
                entry_row = row
            elif basis < -ENTRY_THRESHOLD_BPS:
                position  = "long_kraken"    # Coinbase expensive → long Kraken
                entry_row = row
        else:
            should_exit = abs(basis) < EXIT_THRESHOLD_BPS

            if should_exit or is_last:
                entry_basis = float(entry_row["basis_bps"])

                # Basis compression = profit
                if position == "long_kraken":
                    gross_bps = entry_basis - basis
                else:
                    gross_bps = basis - entry_basis

                net_bps = gross_bps - ROUND_TRIP_BPS
                net_pnl = (balance / 2) * (net_bps / 10_000)
                balance += net_pnl

                holding_min = (
                    (row["timestamp"] - entry_row["timestamp"]).total_seconds() / 60
                )
                trades.append({
                    "symbol":          base,
                    "position":        position,
                    "entry_time":      entry_row["timestamp"],
                    "exit_time":       row["timestamp"],
                    "entry_basis":     round(entry_basis, 2),
                    "exit_basis":      round(basis, 2),
                    "gross_bps":       round(gross_bps, 2),
                    "net_bps":         round(net_bps, 2),
                    "net_pnl_usd":     round(net_pnl, 4),
                    "balance_usd":     round(balance, 4),
                    "holding_min":     round(holding_min, 1),
                    "winner":          net_pnl > 0,
                })
                position  = None
                entry_row = None

    n    = len(trades)
    wins = sum(1 for t in trades if t["winner"])
    sim_results.append({
        "symbol":          base,
        "trade_count":     n,
        "winner_count":    wins,
        "loser_count":     n - wins,
        "win_rate_pct":    round(wins / n * 100, 1) if n else 0.0,
        "net_pnl_usd":     round(balance - STARTING_CAPITAL, 2),
        "return_pct":      round((balance / STARTING_CAPITAL - 1) * 100, 3),
        "avg_holding_min": round(np.mean([t["holding_min"] for t in trades]), 1) if trades else 0.0,
        "ending_balance":  round(balance, 2),
    })
    all_trades.extend(trades)

sim_df    = pd.DataFrame(sim_results).sort_values("net_pnl_usd", ascending=False)
trades_df = pd.DataFrame(all_trades)

print("\n✅ Simulation results (sorted by net P&L):")
display(sim_df)

In [ ]:
# ── Equity curve and trades for best-performing symbol ─────────────────────────

if not sim_df.empty and sim_df.iloc[0]["trade_count"] > 0:
    best_sym    = sim_df.iloc[0]["symbol"]
    best_frame  = pair_frames[best_sym]
    best_trades = trades_df[trades_df["symbol"] == best_sym].copy()

    # ── Equity curve
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(
        x=best_trades["exit_time"],
        y=best_trades["balance_usd"],
        mode="lines+markers",
        name="equity",
        line=dict(color="royalblue"),
    ))
    fig4.add_hline(y=STARTING_CAPITAL, line_dash="dot", line_color="gray")
    fig4.update_layout(
        title=f"{best_sym} (Perps): Equity curve",
        template="plotly_white", height=350, width=1100,
    )
    fig4.show()

    # ── Basis with entry/exit markers
    fig5 = go.Figure()
    fig5.add_trace(go.Scatter(
        x=best_frame["timestamp"],
        y=best_frame["basis_bps"],
        mode="lines",
        name="basis_bps",
        line=dict(width=0.7, color="steelblue"),
    ))
    for level, color, label in [
        (ENTRY_THRESHOLD_BPS,  "green",   f"entry long_kraken ({ENTRY_THRESHOLD_BPS:.0f} bps)"),
        (-ENTRY_THRESHOLD_BPS, "firebrick", f"entry long_coinbase ({-ENTRY_THRESHOLD_BPS:.0f} bps)"),
    ]:
        fig5.add_hline(
            y=level, line_dash="dash", line_color=color,
            annotation_text=label, annotation_position="right",
        )

    entries_kr = best_trades[best_trades["position"] == "long_kraken"]
    fig5.add_trace(go.Scatter(
        x=entries_kr["entry_time"], y=entries_kr["entry_basis"],
        mode="markers", marker=dict(symbol="triangle-up", size=6, color="green"),
        name="entry long_kraken",
    ))
    entries_cb = best_trades[best_trades["position"] == "long_coinbase"]
    fig5.add_trace(go.Scatter(
        x=entries_cb["entry_time"], y=entries_cb["entry_basis"],
        mode="markers", marker=dict(symbol="triangle-down", size=6, color="firebrick"),
        name="entry long_coinbase",
    ))

    fig5.update_layout(
        title=f"{best_sym} (Perps): Basis with entry signals",
        template="plotly_white", height=450, width=1200,
    )
    fig5.show()

    print(f"\nTrade log for {best_sym} (first 30 trades):")
    display(
        best_trades[[
            "position", "entry_time", "exit_time",
            "entry_basis", "exit_basis", "gross_bps",
            "net_bps", "net_pnl_usd", "holding_min", "winner",
        ]].head(30)
    )
else:
    print("No trades executed — entry threshold may be too high relative to observed basis spreads.")
    print(f"Try reducing ENTRY_THRESHOLD_BPS below {ENTRY_THRESHOLD_BPS:.0f} bps.")

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────────

print("=" * 80)
print("CROSS-EXCHANGE PERPETUAL FUTURES ARBITRAGE SUMMARY")
print("=" * 80)
print(f"Window             : {DATE_FROM_ISO[:10]} → {DATE_TO_ISO[:10]}  (2 weeks @ 1m)")
print(f"Symbols analysed   : {list(pair_frames.keys())}")
print(f"Round-trip fees    : {ROUND_TRIP_BPS:.1f} bps  (taker/taker, conservative est.)")
print()

if not mr_df.empty:
    best_mr = mr_df.dropna(subset=["half_life_min"]).sort_values("half_life_min").iloc[0]
    print("Best mean-reversion candidate (fastest basis reversion):")
    print(f"  {best_mr['symbol']}  —  half-life: {best_mr['half_life_min']:.1f} min  "
          f"|  {best_mr['pct_profitable']:.2f}% of time |basis| > fee threshold")
    print()

if not sim_df.empty:
    print(f"Simulation (entry {ENTRY_THRESHOLD_BPS:.0f} bps / exit {EXIT_THRESHOLD_BPS:.0f} bps):")
    for _, row in sim_df.iterrows():
        pnl_str = f"${row['net_pnl_usd']:+.2f}"
        print(f"  {row['symbol']:6s}  trades={row['trade_count']:3d}  "
              f"win={row['win_rate_pct']:.0f}%  PnL={pnl_str:>10s}  "
              f"avg_hold={row['avg_holding_min']:.1f} min")
    print()
    total_pnl = sim_df["net_pnl_usd"].sum()
    print(f"Total P&L across all {len(sim_df)} symbols: ${total_pnl:+.2f}")

print()
print("KEY INSIGHT:")
print(f"  Perpetual futures at {ROUND_TRIP_BPS:.0f} bps round-trip is {172/ROUND_TRIP_BPS:.1f}× cheaper than spot.")
print(f"  Basis spreads typically 10–100 bps; with fees at {ROUND_TRIP_BPS:.0f} bps, viability depends")
print(f"  on basis volatility. If basis regularly exceeds {ENTRY_THRESHOLD_BPS:.0f} bps (entry threshold),")
print("  this strategy is far more viable than spot-to-spot arbitrage.")